In [1]:
import torch
import torchaudio
import numpy as np
import onnxruntime
import whisper
from datasets import load_dataset, Dataset
from huggingface_hub import hf_hub_download
from tqdm import tqdm
from dataclasses import dataclass
from itertools import repeat

from typing import List, Dict, Any
from transformers import WhisperFeatureExtractor

from speech_tokenizer.modeling_whisper import WhisperVQEncoder
from speech_tokenizer.utils import extract_speech_token

@dataclass
class TokenizerConfig:
    mimi_weight_path: str = "tokenizer-e351c8d8-checkpoint125.safetensors"
    mimi_repo: str = "kyutai/moshiko-pytorch-bf16"
    glm_tokenizer_path: str = "THUDM/glm-4-voice-tokenizer"
    cosyvoice_onnx_path: str = "speech_tokenizer_v1.onnx"

class MultiTokenizer:
    def __init__(self, config: TokenizerConfig):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.config = config
        
        # Initialize Mimi using Transformers
        self._init_mimi()
        
        # Initialize GLM Voice
        self._init_glm_voice()
        
        # Initialize CosyVoice
        self._init_cosyvoice()
    
    def _init_mimi(self):
        from mini.models import loaders
        mimi_weight = hf_hub_download(
            self.config.mimi_repo, 
            self.config.mimi_weight_path, 
            local_dir='./checkpoint/mini'
        )
        self.mimi = loaders.get_mimi(mimi_weight, self.device)
        self.mimi.set_num_codebooks(1)
        self.mimi_sample_rate = self.mimi.sample_rate
    
    def _init_glm_voice(self):
        print("Loading GLM Voice model...")
        self.whisper_model = WhisperVQEncoder.from_pretrained(
            self.config.glm_tokenizer_path
        ).eval().to(self.device)
        self.glm_feature_extractor = WhisperFeatureExtractor.from_pretrained(
            self.config.glm_tokenizer_path
        )
    
    def _init_cosyvoice(self):
        print("Loading CosyVoice model...")
        option = onnxruntime.SessionOptions()
        option.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
        option.intra_op_num_threads = 1
        providers = ["CUDAExecutionProvider"]
        self.cosy_session = onnxruntime.InferenceSession(
            self.config.cosyvoice_onnx_path, 
            sess_options=option, 
            providers=providers
        )
    
    def get_mimi_tokens(self, audio_array, sample_rate: int, token_num) -> List[int]:
        wav = torch.from_numpy(audio_array).float()
        if sample_rate != self.mimi_sample_rate:
            wav = torchaudio.transforms.Resample(sample_rate, self.mimi_sample_rate)(wav)
                
        # 添加批次维度
        wav = wav.unsqueeze(1).to(self.device)
        
        with torch.no_grad():
            # 输出 tensor cuda (B, 1, T)
            outputs = self.mimi.encode(wav)
            tokens = outputs.cpu().numpy().astype(np.int32)

            return tokens
    
    # 支持 batch
    def get_glm_tokens(self, audio_array, sample_rate: int, token_num) -> List[int]:
        wav = torch.from_numpy(audio_array).float()
        wav = wav.unsqueeze(1)

        combined_list = list(zip(wav, repeat(sample_rate)))

        # 输出 list
        tokens = extract_speech_token(self.whisper_model, self.glm_feature_extractor, combined_list)
        # 如果输出 numpy
        tokens = np.array(tokens).astype(np.int32)

        return tokens
        
    # 不支持batch
    def get_cosy_tokens(self, audio_array, sample_rate: int, token_num) -> List[int]:
        wav = torch.from_numpy(audio_array).float()

        if sample_rate != 16000:
            wav = torchaudio.transforms.Resample(sample_rate, 16000)(wav)
        
        if wav.shape[1] / 16000 > 30:
            print('Skip: audio longer than 30s')
            return []
        
        feat = whisper.log_mel_spectrogram(wav, n_mels=128)
        speech_token = self.cosy_session.run(
            None, 
            {
                self.cosy_session.get_inputs()[0].name: feat.detach().cpu().numpy(),
                self.cosy_session.get_inputs()[1].name: np.array([feat.shape[2]], dtype=np.int32)
            }
        )[0].flatten().tolist()
        
        return speech_token

class DatasetProcessor:
    def __init__(self, tokenizer_config: TokenizerConfig):
        self.tokenizer = MultiTokenizer(tokenizer_config)
        self.total_duration = 0
        self.processed_files = set()
        self.processed_samples = []
        
    def process_sample(self, sample: Dict[str, Any]) -> Dict[str, Any]:
        """处理单个样本，添加所有tokenizer的tokens"""
        try:
            audio_array = sample['mp3']['array']
            sampling_rate = sample['mp3']['sampling_rate']
            
            # 获取所有tokenizer的tokens
            tokens = {
                'Moshi_mimi': self.tokenizer.get_mimi_tokens(audio_array, sampling_rate),
                'GLM_voice': self.tokenizer.get_glm_tokens(audio_array, sampling_rate),
                'Cosyvoice_tokenizer': self.tokenizer.get_cosy_tokens(audio_array, sampling_rate)
            }
            
            # 创建新样本
            new_sample = {
                'key': sample['key'],
                'url': sample['url'],
                'json': sample['json'],
                'mp3': sample['mp3'],
                'tokens': tokens
            }
            
            return new_sample
        except Exception as e:
            print(f"Error processing sample {sample['key']}: {str(e)}")
            return None
    
    def should_continue(self, sample: Dict[str, Any]) -> bool:
        """检查是否应该继续处理数据"""
        self.total_duration += sample['json']['duration']
        file_id = sample['key'].split('_')[0]
        self.processed_files.add(file_id)
        
        # 检查是否达到限制条件（100小时或10个文件）
        hours_limit = self.total_duration / 3600 < 100
        files_limit = len(self.processed_files) < 10
        
        return hours_limit and files_limit
    
    def process_dataset(self, dataset_path: str = "amphion/Emilia-Dataset"):
        """处理数据集并返回新的数据集"""
        # 加载原始数据集
        dataset = load_dataset(
            dataset_path,
            data_files={"zh": "ZH/*.tar"},
            split="zh",
            streaming=True
        )
        
        for sample in tqdm(dataset, desc="Processing samples"):
            if not self.should_continue(sample):
                break
            
            processed_sample = self.process_sample(sample)
            if processed_sample is not None:
                self.processed_samples.append(processed_sample)
            
            print(f"Progress: {self.total_duration/3600:.2f} hours, "
                  f"{len(self.processed_files)} files, "
                  f"{len(self.processed_samples)} samples")
        
        return Dataset.from_list(self.processed_samples)

/root/miniconda3/envs/GLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset
path = "ZH/ZH-B000000.tar"
dataset = load_dataset("amphion/Emilia-Dataset", data_files={"zh": path}, split="zh")
print(dataset) # here should only shows 90 n_shards instead of 2360

Dataset({
    features: ['__key__', '__url__', 'json', 'mp3'],
    num_rows: 34732
})


In [3]:
# 选择样本
samples = dataset.select([0, 1, 2, 3, 4])

audio_array = [sample['array'] for sample in samples['mp3']]
sampling_rate = 32000

# 找到最长的元素及其长度
max_length = max(len(x) for x in audio_array)

# 对其他特征进行填充
padded_features = [np.pad(feature, (0, max_length - len(feature)), mode='constant') for feature in audio_array]

audio_array = np.stack(padded_features)


duration = [sample['duration'] for sample in samples['json']]

# 计算token数目
duration = np.array(duration)
token_num = np.ceil(duration * 12.5)

In [4]:
samples[0]

{'__key__': 'ZH_B00000_S00000_W000000',
 '__url__': '/root/.cache/huggingface/datasets/downloads/b0f5917a6e070e056c8c12a96d57b19f6e7efbca023279838a85e472c97750c0',
 'json': {'dnsmos': 3.3234,
  'duration': 3.868,
  'id': 'ZH_B00000_S00000_W000000',
  'language': 'zh',
  'speaker': 'ZH_B00000_S00000',
  'text': '能源风口一路护航，你们好。我是张航本行。',
  'wav': 'ZH_B00000/ZH_B00000_S00000/mp3/ZH_B00000_S00000_W000000.mp3'},
 'mp3': {'path': 'ZH_B00000_S00000_W000000.mp3',
  'array': array([-0.00090152, -0.00044956,  0.00080793, ...,  0.00017487,
          0.00025074,  0.00023569]),
  'sampling_rate': 32000}}

In [5]:
config = TokenizerConfig(
    mimi_weight_path = "tokenizer-e351c8d8-checkpoint125.safetensors",
    mimi_repo = "kyutai/moshiko-pytorch-bf16",
    glm_tokenizer_path = "/home/rickc/Projects/LLMs/GLM-4-Voice/THUDM/glm-4-voice-tokenizer",
    cosyvoice_onnx_path = "/home/rickc/Projects/LLMs/CosyVoice/pretrained_models/CosyVoice-300M/speech_tokenizer_v1.onnx"
)

tokenizer = MultiTokenizer(config) 
# 获取所有tokenizer的tokens

tokens = {
    'Moshi_mimi': tokenizer.get_mimi_tokens(audio_array, sampling_rate, token_num),
    'GLM_voice': tokenizer.get_glm_tokens(audio_array, sampling_rate, token_num),
    # 'Cosyvoice_tokenizer': tokenizer.get_cosy_tokens(audio_array, sampling_rate, token_num)
}

Loading GLM Voice model...
Loading CosyVoice model...


2024-11-19 09:51:07.067280550 [E:onnxruntime:Default, provider_bridge_ort.cc:1480 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1193 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.11: cannot open shared object file: No such file or directory

2024-11-19 09:51:07.067313162 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:743 CreateExecutionProviderInstance] Failed to create CUDAExecutionProvider. Please reference https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements to ensure all dependencies are met.
/root/miniconda3/envs/GLM/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/A

In [6]:
# 创建新的数据集
new_dataset = []

for i, sample in enumerate(samples):
    mimi_tokens = tokens['Moshi_mimi'][i].squeeze()[:int(token_num[i])]
    glm_tokens = tokens['GLM_voice'][i][:int(token_num[i])]+10000
    new_sample = {
        'original_key': sample['__key__'],  # 保留原始数据集的key
        'Moshi_mimi': mimi_tokens.tolist(),
        'GLM_voice': glm_tokens.tolist(),
        # 'Cosyvoice_tokenizer': tokens['Cosyvoice_tokenizer'][i][:token_num[i]*4]
    }
    new_dataset.append(new_sample)

In [7]:
new_dataset

[{'original_key': 'ZH_B00000_S00000_W000000',
  'Moshi_mimi': [1952,
   1103,
   1332,
   1678,
   4,
   781,
   1958,
   1709,
   1980,
   76,
   960,
   1824,
   832,
   470,
   195,
   289,
   1491,
   164,
   164,
   1792,
   1792,
   1660,
   1120,
   978,
   1955,
   1633,
   771,
   721,
   1841,
   1792,
   1340,
   1631,
   997,
   451,
   1976,
   1451,
   1887,
   111,
   904,
   904,
   1792,
   1729,
   1626,
   904,
   977,
   1792,
   23,
   1660,
   769],
  'GLM_voice': [22867,
   12154,
   20730,
   25511,
   21848,
   10815,
   18068,
   10335,
   14004,
   18146,
   16192,
   12098,
   12082,
   18816,
   11062,
   15296,
   10017,
   20091,
   25528,
   25669,
   21518,
   25143,
   23964,
   24770,
   17661,
   23011,
   10485,
   12807,
   13951,
   13566,
   18436,
   10492,
   20094,
   18502,
   12467,
   24267,
   22057,
   12964,
   16367,
   14248,
   19120,
   14776,
   19211,
   23889,
   11896,
   16055,
   17793,
   15240,
   17596]},
 {'original_key': '

In [8]:
new_dataset[0]['Moshi_mimi']

[1952,
 1103,
 1332,
 1678,
 4,
 781,
 1958,
 1709,
 1980,
 76,
 960,
 1824,
 832,
 470,
 195,
 289,
 1491,
 164,
 164,
 1792,
 1792,
 1660,
 1120,
 978,
 1955,
 1633,
 771,
 721,
 1841,
 1792,
 1340,
 1631,
 997,
 451,
 1976,
 1451,
 1887,
 111,
 904,
 904,
 1792,
 1729,
 1626,
 904,
 977,
 1792,
 23,
 1660,
 769]

In [9]:
new_dataset[0]['GLM_voice']

[22867,
 12154,
 20730,
 25511,
 21848,
 10815,
 18068,
 10335,
 14004,
 18146,
 16192,
 12098,
 12082,
 18816,
 11062,
 15296,
 10017,
 20091,
 25528,
 25669,
 21518,
 25143,
 23964,
 24770,
 17661,
 23011,
 10485,
 12807,
 13951,
 13566,
 18436,
 10492,
 20094,
 18502,
 12467,
 24267,
 22057,
 12964,
 16367,
 14248,
 19120,
 14776,
 19211,
 23889,
 11896,
 16055,
 17793,
 15240,
 17596]

In [10]:
import json


with open('dataset001.json', 'w') as f:
    json.dump(new_dataset, f)
